Pre processing dei dati


In [2]:
import torch
import numpy as np
np.random.seed(1122)
torch.random.manual_seed(1122)
print(torch.cuda.is_available())

True


In [3]:
path_dataset = "../dataset/food11"
path_modelli = "../modelli"
path_logs = "../logs_food11"
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [4]:
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, ConcatDataset


imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])


# Creiamo i dataset per training, evaluation e validation
train_dataset = datasets.ImageFolder(
    root=os.path.join(path_dataset, "training"),
    transform=transform
)

eval_dataset = datasets.ImageFolder(
    root=os.path.join(path_dataset, "evaluation"),
    transform=transform
)

val_dataset = datasets.ImageFolder(
    root=os.path.join(path_dataset, "validation"),
    transform=transform
)


print("Classi trovate nel training set:", train_dataset.classes)

    # Creazione dei data loader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,num_workers=1,
    pin_memory=True)
eval_loader = DataLoader(eval_dataset, batch_size=32, shuffle=False,num_workers=1,
    pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False,num_workers=1,
    pin_memory=True)

    
# train + eval
combined_dataset = ConcatDataset([train_dataset, eval_dataset])
combined_loader = DataLoader(combined_dataset, batch_size=32, shuffle=True, num_workers=1)

# Stampare immagini e label
for images, labels in combined_loader:
    print("Shape immagini:", images.shape)      # [batch_size, channels, height, width]
    print("Label batch:", labels)
    break

Classi trovate nel training set: ['Bread', 'Dairy product', 'Dessert', 'Egg', 'Fried food', 'Meat', 'Noodles-Pasta', 'Rice', 'Seafood', 'Soup', 'Vegetable-Fruit']
Shape immagini: torch.Size([32, 3, 224, 224])
Label batch: tensor([ 4,  4,  8,  2,  7,  3, 10,  0,  4,  9,  3,  6,  9,  2,  8,  8,  0,  5,
         9,  6,  8,  8,  4,  3,  4,  7,  2,  0,  5,  6,  1,  3])


In [5]:
from collections import Counter

def count_images_per_class(dataset, dataset_name):
    counter = Counter(dataset.targets)
    print(f"Distribuzione nel dataset {dataset_name}:")
    for class_idx, num_images in sorted(counter.items()):
        class_name = dataset.classes[class_idx]
        print(f" - Classe '{class_name}': {num_images} immagini")
    print()

# Calcola distribuzioni sui tre dataset
count_images_per_class(train_dataset, "Training")
count_images_per_class(eval_dataset, "Evaluation")
count_images_per_class(val_dataset, "Validation")

Distribuzione nel dataset Training:
 - Classe 'Bread': 994 immagini
 - Classe 'Dairy product': 429 immagini
 - Classe 'Dessert': 1500 immagini
 - Classe 'Egg': 986 immagini
 - Classe 'Fried food': 848 immagini
 - Classe 'Meat': 1325 immagini
 - Classe 'Noodles-Pasta': 440 immagini
 - Classe 'Rice': 280 immagini
 - Classe 'Seafood': 855 immagini
 - Classe 'Soup': 1500 immagini
 - Classe 'Vegetable-Fruit': 709 immagini

Distribuzione nel dataset Evaluation:
 - Classe 'Bread': 368 immagini
 - Classe 'Dairy product': 148 immagini
 - Classe 'Dessert': 500 immagini
 - Classe 'Egg': 335 immagini
 - Classe 'Fried food': 287 immagini
 - Classe 'Meat': 432 immagini
 - Classe 'Noodles-Pasta': 147 immagini
 - Classe 'Rice': 96 immagini
 - Classe 'Seafood': 303 immagini
 - Classe 'Soup': 500 immagini
 - Classe 'Vegetable-Fruit': 231 immagini

Distribuzione nel dataset Validation:
 - Classe 'Bread': 362 immagini
 - Classe 'Dairy product': 144 immagini
 - Classe 'Dessert': 500 immagini
 - Classe 'Egg

Procedure di train, test e valutazione


In [6]:
class AverageValueMeter():
  def __init__(self):
    self.reset()

  def reset(self):
    self.sum = 0
    self.num = 0

  def add(self,value,num):
    self.sum += value*num
    self.num += num

  def value(self):
    try:
      return self.sum/self.num
    except:
      return None

In [7]:
# definiamo la consueta funzione per ottenere probabilità di test predette dal modello
def test_classifier(model, loader):
  device = "cuda" if torch.cuda.is_available() else "cpu"
  model.to(device)
  predictions, labels = [],[]
  for batch in loader:
    x = batch[0].to(device)
    y = batch[1].to(device)
    output = model(x)
    preds = output.to("cpu").max(1)[1].numpy() # le funzioni numpy girano solo su cpu
    labs = y.to("cpu").numpy()
    predictions.extend(list(preds))
    labels.extend(list(labs))
  return np.array(predictions), np.array(labels)

In [8]:
from sklearn.metrics import (accuracy_score,classification_report,confusion_matrix)

def evaluate_classifier(y_true, y_pred, class_names=None):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    acc = accuracy_score(y_true, y_pred)
    report = classification_report(y_true, y_pred, target_names=class_names, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
    print("=== Classification Report ===")
    print(report)
    #plot_confusion_matrix(cm,class_names)
    # print(cm)
    return report,cm


Finetuning con Squeezenet



In [9]:
from torch import nn
from torchvision.models import squeezenet1_0
from torchvision.models import SqueezeNet1_0_Weights

def get_squeezenet1_model(num_class = 100):
  model = squeezenet1_0(weights = SqueezeNet1_0_Weights)
  num_class = 11
  model.classifier[1] = nn.Conv2d(512,num_class,kernel_size = (1,1), stride = (1,1))
  model.num_classes = num_class
  return model

# carica il modello preaddestrato
model = get_squeezenet1_model()

#model = train_classifier(model, train_loader, eval_loader, exp_name="squeezenet_food11_finetuning", epochs=10, lr=0.001)

#state_dict = torch.load(path_modelli + "/squeezenet_food11_finetuning-7.pth", map_location=device)
#model.load_state_dict(state_dict)

c:\Users\Mannino\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=SqueezeNet1_0_Weights.IMAGENET1K_V1`. You can also use `weights=SqueezeNet1_0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [10]:
#y_pred_squeezenet_food11, y_true_food11 = test_classifier(model, eval_loader)
#evaluate_classifier(y_pred_squeezenet_food11,y_true_food11)

In [11]:
# funzione che supporta anche la ripresa
from torch.optim import SGD
from torch.utils.tensorboard import SummaryWriter
from sklearn.metrics import accuracy_score
from os.path import join
import torch.nn as nn


def train_classifier3(model, train_loader, test_loader, checkpoint = None, exp_name = 'experiment', lr = 0.01, epochs = 10, momentum = 0.99, logdir = 'logs'):
  if checkpoint is not None:
      # Ripristina il modello
      model.load_state_dict(checkpoint['model_state_dict'])

    # Ripristina l'optimizer
      optimizer = SGD(model.parameters(), lr, momentum=momentum)
      optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    # Ripristina global step
      global_step = checkpoint['global_step']
      epoch_start = checkpoint['epoch']
  else:
      optimizer = SGD(model.parameters(), lr, momentum = momentum) # funzione di learning
      global_step = 0
      epoch_start = 0

  criterion = nn.CrossEntropyLoss() # funzione di loss
  loss_meter = AverageValueMeter()
  acc_meter = AverageValueMeter()

  # writer
  writer = SummaryWriter(join(path_logs,exp_name))
  print(writer, " $ ", join(path_logs,exp_name))

  # device
  device = 'cuda' if torch.cuda.is_available() else 'cpu'
  print(device)
  model.to(device)

  # definiamo un dizionario contenente i loader di training e test
  loader = {
      'train': combined_loader,
      'test': val_loader
  }


  for e in range(epoch_start,epochs + epoch_start):
    print(f"Epoch: {e+1} of {epochs + epoch_start}")
    # iteriamo tra due modalità: train e test
    for mode in ['train','test']:
      loss_meter.reset()
      acc_meter.reset()
      model.train() if mode == 'train' else model.eval()
      with torch.set_grad_enabled(mode == 'train'): # abilitiamo i gradienti in training
        for i, batch in enumerate(loader[mode]):
          x = batch[0].to(device) #portiamoli sul device corretto
          y = batch[1].to(device)


          output = model(x)
          l = criterion(output, y)

          # aggiorniamo il global_step
          # conterrà il numero di campioni visti durante il training
          n = x.shape[0]
          global_step += n


          #print(f"global_step = {global_step}")

          if mode == 'train':
              l.backward()
              optimizer.step()
              optimizer.zero_grad()

          acc = accuracy_score(y.to('cpu'),output.to('cpu').max(1)[1])
          loss_meter.add(l.item(),n)
          acc_meter.add(acc,n)

          # loggiamo i risultati iterazione per iterazione solo durante il training
          if mode == 'train':
            writer.add_scalar('loss/train', loss_meter.value(), global_step = global_step)
            writer.add_scalar('accuracy/train', acc_meter.value(), global_step = global_step)

      # una volta finita l'epoca (sia nel caso di training che test, loggiamo le stime finali)
      writer.add_scalar('loss/'+mode, loss_meter.value(), global_step = global_step)
      writer.add_scalar('accuracy/'+mode, acc_meter.value(), global_step = global_step)

   # conserviamo i pesi del modello alla fine di ogni epoca
      #torch.save(model.state_dict(),'%s-%d.pth' % (exp_name,e))
      torch.save({
          'global_step': global_step,
          'model_state_dict': model.state_dict(),
          'optimizer_state_dict': optimizer.state_dict(),
          'epoch' : e+1,
      }, path_modelli +'/checkpoint-%s-%d.pth'%(exp_name,e+1))
  #torch.save(model.state_dict(), path_modelli + f"{exp_name}-{e+1}.pth")

  return model

In [ ]:
# Carica il checkpoint
#checkpoint = torch.load(path_modelli + "/checkpoint-squeezenet_food11_finetuning3-5.pth", map_location=device)
#model.load_state_dict(checkpoint['model_state_dict'])


#y_pred_squeezenet_food11, y_true_food11 = test_classifier(model, val_loader)
#print(accuracy_score(y_true_food11,y_pred_squeezenet_food11))


KeyboardInterrupt: 

In [ ]:
 #ripresa
model = train_classifier3(model, combined_loader, val_loader, exp_name="squeezenet_food11_finetuning3", epochs=10, lr=0.01, logdir = path_logs)

<torch.utils.tensorboard.writer.SummaryWriter object at 0x00000203CDB40FE0>  $  ../logs_food11\squeezenet_food11_finetuning3
cuda
Epoch: 1 of 10
Epoch: 2 of 10
Epoch: 3 of 10
Epoch: 4 of 10
Epoch: 5 of 10
Epoch: 6 of 10
Epoch: 7 of 10
Epoch: 8 of 10
Epoch: 9 of 10
Epoch: 10 of 10


In [ ]:
y_pred_squeezenet_food11, y_true_food11 = test_classifier(model, val_loader)
print(accuracy_score(y_true_food11,y_pred_squeezenet_food11))

0.10994920824619062


***

In [ ]:
from tkinter import simpledialog
import customtkinter
import tkinter
from tkinter import messagebox
from PIL import ImageTk, Image
from tkinter import filedialog as fd
from datetime import datetime
import numpy as np
#import ipynb.fs.full.Esperimenti_su_food11 as il_mio_notebook_module

customtkinter.set_appearance_mode("dark")               # colore tema
customtkinter.set_default_color_theme("dark-blue")

# Seconda parte GUI
# ----------------------------------------------------------------------------
root2 = customtkinter.CTk()
root2.title("Demo Classificazione Cibi e Bevande")
cont = 0
file = ''

def openfilename():
    # open file dialog box to select image
    # The dialogue box has a title "Open"
    filename = fd.askopenfilename(title ='Scegli immagine')
    return filename

def caricaImg():    
    x = openfilename()
    global cont

    if x == '':
        return

    file = x
    cont = cont + 1
    # opens the image

    img_pil = Image.open(x)
    if img_pil.width > 400 or img_pil.height > 400:
        #print("ciao")
        if img_pil.width > img_pil.height:
            #print("ciao")
            width = 400
            height = int( (width * img_pil.height) / img_pil.width)
            resized_img = img_pil.resize((width, height), Image.LANCZOS)
        else:
            #print("ciao")
            height = 400
            width = int( (height * img_pil.width) / img_pil.height)
            resized_img = img_pil.resize((width, height), Image.LANCZOS)
    else:
        resized_img = img_pil.resize((img_pil.width, img_pil.height), Image.LANCZOS)
    ctk_img = customtkinter.CTkImage(img_pil, size=resized_img.size)

        
    # carico immagine nel frame corrispondente
    photo_label = customtkinter.CTkLabel(frame1, image = ctk_img, text="")
    photo_label.place(relx=.5, rely=.5, anchor='center')

def classify():
    print("prova")

def on_closing2():
    if messagebox.askokcancel("Esci", "Sei sicuro di voler uscire?"):
        root2.destroy()
        exit()

root2.protocol("WM_DELETE_WINDOW", on_closing2)

root2.geometry("860x550")        # Definisco dim finestra
w2 = 860
h2 = 550
# get screen width and height
ws = root2.winfo_screenwidth() # width of the screen
hs = root2.winfo_screenheight() # height of the screen

        # calculate x and y coordinates for the Tk root window
x = (ws/2) - (w2/2)
y = (hs/2) - (h2/2)

        # set the dimensions of the screen 
        # and where it is placed
root2.geometry('%dx%d+%d+%d' % (w2, h2, x, y))

        
root2.columnconfigure(0, weight=1)
root2.columnconfigure(1, weight=1)

root2.rowconfigure(0, weight=1)
frame1 = customtkinter.CTkFrame(root2)
frameComandi = customtkinter.CTkFrame(root2)

button1 = customtkinter.CTkButton(master=frame1, text="Aggiungi immagine", command=lambda:caricaImg()) # inserisco bottone alla finestra
button1.place(relx=0.5, rely=0.5, anchor='center')

label = customtkinter.CTkLabel(master=frameComandi, text="Pannello dei comandi", font=('Arial', 20))  # aggiungo etichetta alla finestra 
label.place(relx=0.5, rely=0.15, anchor='center')


comparison = customtkinter.CTkButton(master=frameComandi, text="Cambia immagine", command=lambda:caricaImg()) # inserisco bottone alla finestra
comparison.place(relx=0.5, rely=0.3, anchor='center')

classificaImmagine = customtkinter.CTkButton(master=frameComandi, text="Classifica Immagine", command=lambda:classify())
classificaImmagine.place(relx=0.5, rely=0.4, anchor='center')
        
frame1.grid(column=0, row=0, sticky='ewns', padx = 20)
frameComandi.grid(column=1, row=0, sticky='ewns', padx=20)
        
root2.mainloop()



# ----------------------------------------------------------------------------

